# Kling Video O1 Video-to-Video Edit API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Kling Video O1 (Video-to-Video Edit)** 模型 API。

## 模型简介

Kling Video O1 Video-to-Video Edit 以输入视频为**编辑基底**直接修改视频内容，例如背景替换、风格迁移等。

- **视频编辑**：基于 `video_url` 指定的原始视频进行 AI 编辑
- **风格参考**：可通过 `image_urls` 传入风格参考图
- **主体参考**：通过 `elements` 传入主体参考

**API 端点**：`fal-ai/kling-video/o1/{mode}/video-to-video/edit`

**Mode 选项**：`standard` / `std`（720P）、`pro`（1080P）

**注意**：O1 **不支持** `4k` 模式；`prompt` 和 `video_url` 为必填参数

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

## 2. 基础用法 — 视频编辑

七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    return fal_client.result(endpoint, request_id)


# 基础调用示例：视频编辑
result = run_fal_task(
    "fal-ai/kling-video/o1/standard/video-to-video/edit",
    arguments={
        "prompt": "把背景替换成日落时的沙滩",
        "video_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-o3/pro-v2v-reference/video_reference.mp4",
        "keep_audio": True,
    },
)

print(result)

In [ ]:
# 展示生成的视频
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 编辑指令，描述对视频的修改意图 |
| `video_url` | string | *必填* | 待编辑的原始视频 URL（MP4/MOV，3-10s） |
| `image_urls` | list[string] | 可选 | 风格参考图片 URL 列表 |
| `elements` | list | 可选 | 主体参考列表 |
| `keep_audio` | boolean | 可选 | 是否保留原视频音频 |

## 3. Pro 模式示例

In [ ]:
# Pro 模式（1080P）视频编辑
result_pro = run_fal_task(
    "fal-ai/kling-video/o1/pro/video-to-video/edit",
    arguments={
        "prompt": "将画面风格改为赛博朋克霓虹灯效果",
        "video_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-o3/pro-v2v-reference/video_reference.mp4",
        "keep_audio": True,
    },
)

video_url = result_pro["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制，可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "fal-ai/kling-video/o1/standard/video-to-video/edit",
    arguments={
        "prompt": "把背景替换成日落时的沙滩",
        "video_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-o3/pro-v2v-reference/video_reference.mp4",
        "keep_audio": True
    },
)

request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "fal-ai/kling-video/o1/standard/video-to-video/edit",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "fal-ai/kling-video/o1/standard/video-to-video/edit",
    request_id,
)

video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. Schema 参考
### Input Schema

```json
{
  "prompt": "string (必填)",
  "video_url": "string (必填, 待编辑视频 URL, MP4/MOV, 3-10s)",
  "image_urls": ["string (可选, 风格参考图)"],
  "elements": ["object (可选, 主体参考列表)"],
  "keep_audio": "boolean (可选)"
}
```

### Output Schema

```json
{
  "video": {
    "file_name": "string",
    "url": "string",
    "content_type": "string",
    "file_size": "number"
  }
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [Kling Video O1 模型页面](https://fal.ai/models/fal-ai/kling-video/o1/standard/video-to-video/edit)